In [ ]:
!pip install asyncua
import asyncio
import json
import csv
from datetime import datetime
from pathlib import Path
from typing import Dict, Optional
from asyncua import Client
import pandas as pd
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


class Config:
    SERVER_URL = "opc.tcp://192.168.1.100:4840/freeopcua/server/"
    USERNAME = "admin"
    PASSWORD = "Smart4.0@2026"
    INTERVALO_COLETA = 5
    PASTA_DADOS = Path("dados_coletados")
    PASTA_LOG = Path("logs")
    PASTA_DADOS.mkdir(exist_ok=True)
    PASTA_LOG.mkdir(exist_ok=True)


class ColetorSmart40:
    def __init__(self):
        self.client = None
        self.dados_historicos = []
        self.nodes_map = {}

    async def conectar(self) -> bool:
        try:
            self.client = Client(url=Config.SERVER_URL)
            self.client.set_user(Config.USERNAME)
            self.client. set_password(Config.PASSWORD)
            await self.client.connect()
            logger.info("✅ Conectado ao servidor Smart 4.0")
            await self._mapear_nos()
            return True
        except Exception as e:
            logger.error(f"❌ Erro na conexão: {e}")
            return False

    async def _mapear_nos(self):
        try:
            objects = self.client.nodes.objects
            fabrica = await objects.get_child(["2:Objects", "2:Fabrica"])
            celula = await fabrica.get_child(["2:Celula_Smart_01"])

            self.nodes_map = {
                "producao_total": await celula.get_child(["2:ProducaoTotal"]),
                "producao_hora": await celula.get_child(["2:ProducaoHora"]),
                "temperatura": await celula.get_child(["2:Temperatura"]),
                "pressao": await celula.get_child(["2:Pressao"]),
                "qualidade": await celula.get_child(["2:Qualidade"]),
                "status": await celula.get_child(["2:StatusMaquina"]),
                "temp_limite": await celula.get_child(["2:TemperaturaLimite"]),
            }
            historico = await celula.get_child(["2:Historico"])
            self.nodes_map["ultimas_temp"] = await historico.get_child(["2:UltimasTemperaturas"])
            self.nodes_map["reiniciar"] = await celula.get_child(["2:ReiniciarMaquina"])
            self.nodes_map["definir_limite"] = await celula.get_child(["2:DefinirTemperaturaLimite"])
            logger.info(f"✅ Mapeados {len(self.nodes_map)} nós")
        except Exception as e:
            logger.error(f"❌ Erro no mapeamento: {e}")

    async def coletar_dados(self) -> Optional[Dict]:
        dados = {"timestamp": datetime.now().isoformat(), "dados": {}, "alertas": []}
        try:
            for nome, node in self.nodes_map.items():
                if nome in ["reiniciar", "definir_limite"]:
                    continue
                try:
                    valor = await node.read_value()
                    dados["dados"][nome] = valor
                    if nome == "temperatura" and valor > 70:
                        dados["alertas"].append({"tipo": "ALTA_TEMPERATURA", "valor": valor})
                    if nome == "qualidade" and valor < 85:
                        dados["alertas"].append({"tipo": "BAIXA_QUALIDADE", "valor": valor})
                except Exception:
                    dados["dados"][nome] = None
            self.dados_historicos.append(dados)
            if len(self.dados_historicos) > 1000:
                self.dados_historicos.pop(0)
            return dados
        except Exception as e:
            logger.error(f"❌ Erro na coleta: {e}")
            return None

    async def coletar_continuamente(self, duracao: int = None):
        logger.info(f"🔄 Iniciando coleta contínua (intervalo: {Config.INTERVALO_COLETA}s)")
        inicio = datetime.now()
        contador = 0
        try:
            while True:
                if duracao and (datetime.now() - inicio).seconds > duracao:
                    break
                contador += 1
                dados = await self.coletar_dados()
                if dados and contador % 10 == 0:
                    logger.info(f"📊 Coleta #{contador}: T={dados['dados'].get('temperatura')}°C | Q={dados['dados'].get('qualidade')}%")
                if contador % 20 == 0:
                    await self.salvar_em_csv()
                await asyncio.sleep(Config.INTERVALO_COLETA)
        except KeyboardInterrupt:
            logger.info("⏹️ Interrompido")
        finally:
            await self.salvar_em_csv()
            logger.info(f"✅ Coleta finalizada. Total: {contador} coletas")

    async def reiniciar_maquina(self, maquina_id: str = "Smart_01"):
        try:
            resultado = await self.nodes_map["reiniciar"].call(maquina_id)
            logger.info(f"🔄 Máquina reiniciada: {resultado}")
            return resultado
        except Exception as e:
            logger.error(f"❌ Erro ao reiniciar: {e}")

    async def definir_limite_temp(self, limite: float):
        try:
            resultado = await self.nodes_map["definir_limite"].call(limite)
            logger.info(f"🌡️ Limite definido: {resultado}")
            return resultado
        except Exception as e:
            logger.error(f"❌ Erro ao definir limite: {e}")

    async def salvar_em_csv(self):
        if not self.dados_historicos:
            return
        try:
            df = pd.DataFrame([{"timestamp": d["timestamp"], **d["dados"]} for d in self.dados_historicos])
            nome = Config.PASTA_DADOS / f"dados_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
            df.to_csv(nome, index=False)
            logger.info(f"💾 CSV salvo: {nome}")
        except Exception as e:
            logger.error(f"❌ Erro ao salvar CSV: {e}")

    async def salvar_em_json(self):
        if not self.dados_historicos:
            return
        try:
            nome = Config.PASTA_DADOS / f"ultima_coleta_{datetime.now().strftime('%Y%m%d')}.json"
            with open(nome, 'w') as f:
                json.dump(self.dados_historicos[-1], f, indent=2)
            logger.info(f"💾 JSON salvo: {nome}")
        except Exception as e:
            logger.error(f"❌ Erro ao salvar JSON: {e}")

    def analisar_historico(self) -> Dict:
        if not self.dados_historicos:
            return {"erro": "Nenhum dado"}
        df = pd.DataFrame([{"timestamp": d["timestamp"], **d["dados"]} for d in self.dados_historicos])
        return {
            "total_coletas": len(df),
            "temperatura_media": df["temperatura"].mean(),
            "qualidade_media": df["qualidade"].mean(),
            "producao_total": df["producao_total"].iloc[-1] if "producao_total" in df else 0,
            "status_mais_comum": df["status"].mode().iloc[0] if "status" in df else "N/A"
        }

    async def desconectar(self):
        if self.client:
            await self.client.disconnect()
            logger.info("🔌 Desconectado")


async def main():
    coletor = ColetorSmart40()
    if not await coletor.conectar():
        return
    try:
        # Coleta única
        dados = await coletor.coletar_dados()
        if dados:
            print(json.dumps(dados, indent=2, default=str))

        # Coleta contínua por 30 segundos
        await coletor.coletar_continuamente(duracao=30)

        # Análise
        analise = coletor.analisar_historico()
        print("\n📈 Análise:", json.dumps(analise, indent=2))

        # Comandos
        await coletor.definir_limite_temp(75.0)
        # await coletor.reiniciar_maquina("Smart_01")  # descomente para testar

        # Salvar
        await coletor.salvar_em_csv()
        await coletor.salvar_em_json()

    except KeyboardInterrupt:
        logger.info("⏹️ Interrompido")
    finally:
        await coletor.desconectar()


if __name__ == "__main__":
    await main()

ERROR:__main__:❌ Erro na conexão: 
